# 02 · Intelligent Agents: Space Administrator and Outage

| | Space Administrator Agent | Outage Agent |
|---|---|---|
| Purpose | Find and assess suitable space; support assignment review | Interpret outages, assess impact, coordinate stakeholders and timing |
| Never does | Book, reserve or assign space | Send messages, approve outages, edit TRIRIGA |
| Uses the model for | Functional fit and "verify actual use" conditions | Rooms recorded as non-lab that look lab-like |

Both agents are deterministic, tool-using workflows (no external LLM: personnel data stays local). Each tool call is
permission-checked, bounded, timed and logged. The agents talk only through the coordinator. Everything below imports
the application modules; nothing is re-implemented.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import logging, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 70)
from app.logging_setup import setup_logging
setup_logging(logging.WARNING)          # keep notebook output focused on results
NAVY, RUST = "#25508F", "#C0652B"       # validated two-series palette used by the app
from datetime import datetime
from app.agents.coordinator import build_system
system = build_system(as_of=datetime(2026, 11, 22))      # fixed reference date → reproducible output
for agent in (system.space, system.outage):
    d = agent.describe()
    print(f"{d['name']:<12} tools={len(d['tools'])}  locked actions={list(d['locked_actions'])}")
    for o in d["objectives"]: print("   •", o)

space_admin  tools=13  locked actions=['send_email', 'write_tririga', 'assign_space', 'approve_outage']
   • Identify candidate spaces that fit a documented need (function, area, seats, department).
   • Distinguish recorded vacancy from actual availability and flag planning reservations.
   • Use the space-function model to flag records whose recorded class may not reflect actual use.
   • Prepare verification questions and the approval path; never assign or book space.
outage       tools=13  locked actions=['send_email', 'write_tririga', 'assign_space', 'approve_outage']
   • Interpret outage notices and operational emails; reconstruct the current confirmed scope and timing.
   • Separate confirmed impacts, potential impacts, preliminary concerns and explicitly unaffected services.
   • Identify Natural Sciences departments affected and their formal approvers, coordination contacts and recipients.
   • Review conflicts and schedule coverage; propose alternative windows when conflicts

## 1 · How the agents access the deep learning model

The model is one tool among many. Its output is evidence class **Inference**; the agent decides what to do with it.

In [2]:
sid = system.predictor.table.query("verify_flag and recorded_group == 'Office'").index[0]
r = system.space.call("model_space_assessment", space_ids=[sid], function="Office")
print("status:", r.status, "| duration:", r.duration_ms, "ms")
print("model output:", r.data[sid]["predicted_group"], r.data[sid]["confidence"], "| verify flag:", r.data[sid]["verify_flag"])
print("evidence:", r.evidence[0].classification, "→", r.evidence[0].claim)

status: success | duration: 1.1 ms
model output: Residential 0.998 | verify flag: True
evidence: inferred → Model infers Residential (100%) for Graduate Apartments · 01 · 100, recorded as Office. Recorded class may not reflect actual use.


## 2 · Space Administrator Agent: an office close to a professor

James's scenario: *"I need a new room for Professor X's new hire."* The agent finds Professor X in the people records, takes
the recorded office as the reference, and ranks candidates by **proximity** (same floor and nearby room numbers first, then
the same building), together with function, area, occupancy, department and planning criteria. The agent resolves the department, finds compatible records,
applies hard constraints (area, recorded capacity), scores six weighted criteria and reports **suitability** and **evidence
coverage** separately. Recorded vacancy is a note, never permission.

In [3]:
from app.agents.schemas import SpaceRequest
req = SpaceRequest(request_type="new_office", near="Bonnie Bartel", function="Office")
space = system.run_space(req)
print(space["summary"])
print([q["value"] for q in space["requirements"] if q["field"] == "Close to"])
cand = pd.DataFrame(space["candidates"])[["room_key", "area", "status", "suitability", "evidence_coverage"]]
cand.insert(1, "distance", [c["proximity"]["label"] for c in space["candidates"]])
cand["model"] = [c["model"]["predicted_group"] if c["model"] else None for c in space["candidates"]]
display(cand)
print("Verification questions:", *space["verification_questions"], sep="\n  - ")
print("Required approvals:", *space["required_approvals"], sep="\n  - ")

8 candidates ranked from 263 compatible records: 4 potentially suitable, 4 conditional.
['Bonnie Bartel (bartel) · George R. Brown · 03 · W300D']


,room_key,distance,area,status,suitability,evidence_coverage,model
0,George R. Brown · 03 · W300B,Same floor · W300,213.6,potentially_suitable,0.939,1.0,Research Lab
1,George R. Brown · 02 · W200H,1 floor away,124.4,potentially_suitable,0.853,1.0,Office
2,George R. Brown · 02 · W200J,1 floor away,124.4,potentially_suitable,0.837,1.0,Office
3,George R. Brown · 02 · W200D,1 floor away,124.4,conditional,0.795,1.0,Research Lab
4,George R. Brown · 01 · W105,2 floors away,134.2,potentially_suitable,0.790,1.0,Research Lab
5,George R. Brown · 02 · W200M,1 floor away,131.7,conditional,0.787,1.0,Research Lab
6,George R. Brown · 02 · E200J,1 floor away,190.5,conditional,0.771,1.0,Office
7,George R. Brown · 01 · W102,2 floors away,134.3,conditional,0.766,1.0,Research Lab


Verification questions:
  - Can the department confirm actual current use of the top candidates (walkthrough if needed)?
Required approvals:
  - Affected department (actual use and needs)
  - School leadership
  - Space Management (allocation change)


## 3 · Outage Agent: an emergency steam outage (real ABL thread)

The input is the whole ABL correspondence file, which mixes several incidents. The agent isolates the current incident,
reconstructs the timeline, and keeps the Oct 22 *preliminary* concerns (heating, hot water) apart from the confirmed scope
(autoclaves only), exactly as James described.

In [4]:
from app.agents.schemas import OutageRequest
from app.scenarios import resolve_payload, scenarios
abl = next(s for s in scenarios() if s.id == "OUT-ABL-STEAM")
out = system.run_outage(OutageRequest(**resolve_payload(abl, system.corpus)))
print(out["summary"]); print("messages set aside as other incidents:", out["incident"]["set_aside"])
display(pd.DataFrame(out["timeline"])[["date", "type", "sender", "superseded"]])
rows = [(group, i["statement"][:90], i["classification"], i.get("note", "")) for group, items in out["impacts"].items() for i in items]
pd.DataFrame(rows, columns=["register", "statement", "evidence", "note"])

SR-1118242 · Steam · ABL · Oct 27 08:00–12:00. 2 confirmed and 2 potential impact items, 0 explicit exclusion(s). 1 Natural Sciences department(s); 0 recorded conflict(s).
messages set aside as other incidents: 28


,date,type,sender,superseded
0,2026-10-22T08:23:03,preliminary,Jose Presas <jap20@rice.edu>,False
1,2026-10-22T09:38:00,update,James Delgado <ljd1@rice.edu>,False
2,2026-10-22T09:54:07,update,Juan Sanchez <cjds@rice.edu>,False
3,2026-10-27T08:13:39,update,Jose Presas <jap20@rice.edu>,False
4,2026-10-27T08:27:27,confirmed,TRIRIGA Notification <TRIRIGA-ServiceCenter@rice.edu>,False


,register,statement,evidence,note
0,confirmed,Expect autoclaves to be out of service for the duration of the out...,confirmed,
1,confirmed,Steam impacted,confirmed,
2,potential,97 recorded laboratory spaces in scope may depend on Steam,recorded,
3,potential,8 non-lab records with a lab-like profile (model): Biology · 02 · ...,inferred,
4,preliminary_superseded,Autoclaves will be out of service,historical,Later confirmed in current scope
5,preliminary_superseded,Building comfort heating will be affected,historical,Not in the confirmed scope
6,preliminary_superseded,Domestic hot water will be affected.,historical,Not in the confirmed scope


## 4 · Does the outage conflict with anything? (real BRK thread)

James's most important check: *will this outage time conflict with an event?* The BRK emails propose elevator work for
"Wednesday, November 25" (a date in prose, no time). A different email says that date starts the Thanksgiving recess. The
agent reads the date, checks it against the events calendar (dated events from the correspondence, imported calendars,
manual entries and times James has blocked), reports the conflict and suggests conflict-free times. If James says "not
then", blocking a time makes every later check flag it.

In [5]:
brk = next(s for s in scenarios() if s.id == "OUT-BRK-ELEVATOR")
o2 = system.run_outage(OutageRequest(**resolve_payload(brk, system.corpus)))
sch = o2["scheduling"]
print("proposed:", o2["outage"]["proposed_start"][:10], "| date only:", o2["outage"]["date_only"], "| duration:", o2["outage"]["duration_hours"], "h")
print("verdict:", sch["verdict"]); display(pd.DataFrame(sch["conflicts"])[["title", "building", "source"]])
display(pd.DataFrame(sch["alternatives"])[["start", "end", "rationale"]])
blocked = system.outage.events.add("Symposium (James: no outage)", datetime(2026, 11, 28, 0), datetime(2026, 11, 29, 0),
                                   "Brockman Hall", "blocked", "blocked")
again = system.outage.review_window(["Brockman Hall"], datetime(2026, 11, 28, 6), datetime(2026, 11, 28, 14))
print("James blocks Nov 28 → re-check verdict:", again["verdict"], "| next suggestion:", again["alternatives"][0]["start"])
system.outage.events.delete(blocked["id"])

proposed: 2026-11-25 | date only: True | duration: 8.0 h
verdict: conflict


,title,building,source
0,"Thanksgiving recess (per email from Jose Presas, Nov 03)",Campus-wide,correspondence


,start,end,rationale
0,2026-11-28T06:00:00,2026-11-28T14:00:00,Saturday morning (weekend still requires departmental review)
1,2026-12-05T06:00:00,2026-12-05T14:00:00,Saturday morning (weekend still requires departmental review)
2,2026-12-12T06:00:00,2026-12-12T14:00:00,Saturday morning (weekend still requires departmental review)


James blocks Nov 28 → re-check verdict: conflict | next suggestion: 2026-12-05T06:00:00


True

## 5 · Agent interaction: the two agents exchange information

Temporary space is rarely needed, so it is **on request only** (`cooperate=True`, or a button in the app). The Outage
Agent sends a `temporary_space_request` through the coordinator; the Space Administrator Agent runs its own workflow and
replies with candidates outside the affected building. Neither agent can call the other's tools.

In [6]:
out = system.run_outage(OutageRequest(**resolve_payload(abl, system.corpus)), cooperate=True)
for m in out["messages"]:
    print(f"{m['sender']} → {m['recipient']} · {m['intent']}")
reply = out["space_support"]["reply"]["payload"]
display(pd.DataFrame(reply["candidates"]))
print(reply["note"])

outage → space_admin · temporary_space_request
space_admin → outage · temporary_space_options


,space_id,building,area,space_class,status,suitability,evidence_coverage,allocation
0,Keck · 03 · 327E,Keck,1259.6,250 - Research/Nonclass Laboratory,conditional,0.975,0.8,BioSciences
1,Keck · 03 · 327W,Keck,1284.2,250 - Research/Nonclass Laboratory,conditional,0.975,0.8,BioSciences
2,BRC · 04 · 490A,BRC,199.3,255 - Research/Nonclass Laboratory Service,conditional,0.975,0.8,BioSciences
3,BRC · 04 · 490B,BRC,626.2,250 - Research/Nonclass Laboratory,conditional,0.975,0.8,BioSciences
4,BRC · 04 · 490B1,BRC,146.4,255 - Research/Nonclass Laboratory Service,conditional,0.975,0.8,BioSciences


Temporary options are candidates for review, not assignments.


## 6 · Results: affected people, stakeholders and the drafts for approval

In [7]:
st = pd.DataFrame([{"department": d["department"], "formal approver": d["formal_approver"]["status"],
                    "coordination contacts": ", ".join(c["name"] for c in d["coordination_contacts"]) or "none on record",
                    "recipients": d["notification_recipients"]["count"]} for d in out["stakeholders"]["departments"]])
display(st)
people = out["affected_people"]
print(people["scope_rule"]); print(f"{people['total']} recorded people affected, {people['natural_sciences']} in Natural Sciences")
display(pd.DataFrame(people["people"])[["name", "email", "department", "room", "reason"]].head(6))
for d in out["communication_drafts"]:
    print(f"\n=== {d['title']} · {d['status']}"); print(d["body"][:700])
print("\nAttempt to send →", system.outage.call("send_email").status)

,department,formal approver,coordination contacts,recipients
0,BioSciences,not_on_record,Juan Sanchez,23


Recorded occupants of laboratory or lab-like rooms that depend on Steam
1 recorded people affected, 1 in Natural Sciences


,name,email,department,room,reason
0,Martha Guerra,mcg3@rice.edu,BioSciences,Biology · 03 · 370,Works in Biology · 03 · 370 (recorded laboratory)


2026-11-23 00:10:58.725 | WARNING | run=- | fai.agents.outage | locked action refused tool=send_email



=== Coordination with departments · DRAFT — NOT SENT
Subject: Coordination: SR-1118242 at M.D. Anderson Biological Laboratories

Hello,

Facilities has scheduled work affecting Steam at M.D. Anderson Biological Laboratories, Tuesday Oct 27, 2026 08:00 to 12:00.

What to expect:
- Expect autoclaves to be out of service for the duration of the outage.

Could you confirm whether any experiments, deliveries, classes or accessibility needs in your areas conflict with this window?
- Some rooms recorded as non-lab look lab-like to the model. Do any host equipment that depends on the affected systems?

Thank you,
James

=== Notice to affected people · DRAFT — NOT SENT
Subject: Building notice: Steam outage at M.D. Anderson Biological Laboratories, Tuesday Oct 27, 2026 08:00 to 12:00

Hello,

Please be aware of upcoming work at M.D. Anderson Biological Laboratories.

When: Tuesday Oct 27, 2026 08:00 to 12:00
Systems: Steam
Why: This outage is necessary for the plant operation team to repair a 

## 7 · Evaluation, cooperation and competition

* **Scenario evaluation** (`app/evaluation.py`): binary checks written from James's documented examples plus governance and
  safety checks (locked actions, permission boundaries, an injected instruction inside a notice).
* **Cooperation** (`app/experiments.py`): every recorded ≥4-hour lab/occupancy outage, run with and without the Space agent.
* **Competition**: three strategies for choosing records to verify (rules, model, combined) on 1,000 injected class errors.

In [8]:
from app.evaluation import evaluate_agents
from app.experiments import competition_experiment, cooperation_experiment
ev = evaluate_agents(system)
print(f"Scenario checks passed: {ev['passed']}/{ev['total']}")
display(pd.DataFrame([{"scenario": s["title"], "passed": f"{s['passed']}/{s['total']}"} for s in ev["scenarios"]]))
coop = cooperation_experiment(system)["summary"]
display(pd.Series(coop, name="cooperation").to_frame())
comp = competition_experiment(system)
display(pd.DataFrame(comp["results"]).T[["precision", "recall", "f1", "flags"]])
print("best strategy:", comp["winner"])

2026-11-23 00:11:00.046 | WARNING | run=- | fai.agents.outage | locked action refused tool=send_email


2026-11-23 00:11:00.046 | WARNING | run=- | fai.agents.outage | permission denied tool=rank_candidate_spaces


Scenario checks passed: 67/67


,scenario,passed
0,Planned exhaust reduction · OES,9/9
1,Emergency steam repair · ABL,10/10
2,Utility power transfer · BRC and others,7/7
3,Elevator out of order · GRB,5/5
4,Elevator pit maintenance · Keck,6/6
5,Elevator work during Thanksgiving recess · BRK,7/7
6,Office near a faculty member · new hire,4/4
7,New faculty office · Chemistry,4/4
8,Research lab for a new program · BioSciences,5/5
9,Vacancy review · Natural Sciences,4/4


,cooperation
outages_evaluated,19.000
with_natural_sciences_departments,19.000
solo_share_with_options,0.000
coop_share_with_options,0.842
constraint_compliance,1.000
mean_suitability,0.948
mean_coverage,0.825
mean_added_latency_ms,170.900


,precision,recall,f1,flags
A_rule_based,0.978,0.363,0.530,371.0
B_model_based,0.956,0.876,0.914,916.0
C_combined,0.952,0.935,0.943,982.0


best strategy: C_combined


## 8 · Limitations

* No approver roster, class schedule or maintenance calendar is in the supplied data, and the events calendar only holds what
  the emails mention plus what the team imports or adds. Gaps are reported as gaps; the agents never assume "no conflict".
* Department contacts come only from what the emails show and are marked *candidate*.
* Dates in prose are read as whole days when no time is given; the agent asks for the hours.
* Proximity is estimated from building, floor and room numbering, not measured walking distance (floor plans are images).
* The model generalizes poorly to buildings it has never seen; competition errors are synthetic and easier than real drift.
* Agents are rule-based workflows; an LLM could later draft richer text behind the same tool and permission boundaries.